# 三步质控快速版

奇偶深度校验 → 行星半径合理性 → NASA 档案交叉比对，**一步完成即时写 CSV**。

**提速手段**（相对分开运行的三个 notebook）：
- 奇偶校验：BLS 降分辨率 + 多扇区拼接 + 下载缓存
- 半径校验：TIC 星表**批量查询**（一次 `query_criteria` 替代 N 次 `query_object`）
- 交叉比对：缓存 + 名称/TIC ID 双键

**输出**：`vetting_result.csv`，列结构与之前 `nasa_crossmatch_result.csv` 完全一致。

## 0. 安装依赖

In [1]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与加载数据

In [2]:
import os, warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)

# ==================== 参数配置 ====================
ODD_EVEN_THRESHOLD   = 3.0          # σ > 3 -> 奇偶不通过
MAX_RP_GENERAL       = 2.5          # Rjup，通用行星半径上限
MAX_RP_MDWARF        = 2.0          # Rjup，M 矮星更严上限
R_SUN_TO_RJUP        = 9.7311       # 1 Rsun = 9.73 Rjup
R_JUP_TO_REARTH      = 11.2089      # 1 Rjup = 11.21 Rearth
STITCH_ALL_SECTORS   = True         # 拼接多扇区 -> 更多凌日
TESS_CACHE_DIR       = 'tess_cache'
OUTPUT_CSV           = 'vetting_result.csv'

os.makedirs(TESS_CACHE_DIR, exist_ok=True)

# ==================== 输入 ====================
_PATHS = ['outputs/master_candidates_500.csv',
          'master_candidates.csv',
          'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])

df = pd.read_csv(INPUT_CSV)
ORIG_DF = df.copy()                          # Person1 原始快照，末尾校验用

# Person1 的 8 个原始列
P1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
           'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

# 最终输出列顺序（与 nasa_crossmatch_result.csv 一致）
FINAL_COLS = P1_COLS + [
    'depth_odd_ppm', 'depth_even_ppm', 'n_transits',
    'odd_even_sigma', 'odd_even_reason', 'odd_even_pass',
    'TIC_ID', 'Stellar_Radius_Rsun', 'Teff_K', 'Spectral_Type',
    'Rp_Rjup', 'Rp_Rearth', 'radius_reason', 'planet_radius_plausible',
    '已知/新候选', 'match_source']

print(f'已加载 {INPUT_CSV}  ->  {len(df)} 个候选')
df.head()

已加载 outputs/master_candidates_500.csv  ->  500 个候选


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive
0,TIC 293694800,61,3.31785,0.06627,380.4,5.40,6.98,True
1,TIC 156919710,6,6.52283,0.14106,1230.7,2.95,6.76,True
2,TIC 232050760,7,18.93983,0.46561,1575.2,1.10,6.18,True
3,TIC 157077690,6,2.17228,0.04949,1119.3,6.00,15.63,False
4,TIC 232041402,6,17.04370,0.52106,1528.9,6.00,9.14,False


## 2. 校验一：奇偶深度检验（快速版）

In [3]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def _get_lc(target, sector):
    '''下载光变曲线(带缓存)，优先 SPOC + 多扇区拼接。'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    search = None
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'author': 'SPOC'},
               {'sector': int(sector)},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    if search is None:
        _LC_CACHE[key] = None
        return None
    lc = None
    try:
        if STITCH_ALL_SECTORS and len(search) > 1:
            lc = search.download_all(download_dir=TESS_CACHE_DIR).stitch()
        else:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        try:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
        except Exception:
            lc = None
    _LC_CACHE[key] = lc
    return lc

def odd_even_check(target, sector, period, duration_hr):
    '''返回 {depth_odd, depth_even, n_transits, sigma, reason}。'''
    out = {'depth_odd': np.nan, 'depth_even': np.nan, 'n_transits': 0,
           'sigma': np.nan, 'reason': ''}
    lc = _get_lc(target, sector)
    if lc is None:
        out['reason'] = 'no_lc'
        return out
    try:
        lc = lc.remove_nans().normalize().flatten(window_length=401)
        t = np.asarray(lc.time.value, dtype=float)
        f = np.asarray(lc.flux.value, dtype=float)
        g = np.isfinite(t) & np.isfinite(f)
        t, f = t[g], f[g]
        if len(t) < 200:
            out['reason'] = f'few_cad({len(t)})'
            return out
        mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
        scl = 1.4826 * mad if mad > 0 else np.nanstd(f)
        dy = np.full(len(t), scl if scl > 0 else 1e-3)
        bls = BoxLeastSquares(t, f, dy=dy)
        p0 = float(period)
        ps = np.linspace(p0 * 0.999, p0 * 1.001, 300)
        d0 = max(float(duration_hr), 0.5) / 24.0
        ds = np.unique(np.clip(np.array([d0 * 0.6, d0, d0 * 1.4]),
                               0.5 / 24.0, 8.0 / 24.0))
        res = bls.power(ps, ds, objective='snr')
        j = int(np.argmax(res.power))
        st = bls.compute_stats(float(res.period[j]),
                              float(res.duration[j]),
                              float(res.transit_time[j]))
        ntr = int(np.sum(np.atleast_1d(st['per_transit_count']) > 0))
        out['n_transits'] = ntr
        do, eo = float(st['depth_odd'][0]), float(st['depth_odd'][1])
        de, ee = float(st['depth_even'][0]), float(st['depth_even'][1])
        out['depth_odd'] = do * 1e6
        out['depth_even'] = de * 1e6
        comb = np.sqrt(eo ** 2 + ee ** 2)
        if not np.isfinite(comb) or comb <= 0:
            out['reason'] = f'err_undef(n={ntr})'
            return out
        out['sigma'] = abs(do - de) / comb
        out['reason'] = 'ok'
        return out
    except Exception as e:
        out['reason'] = f'ex:{type(e).__name__}'
        return out

In [4]:
print('=== 校验一：奇偶深度 ===')
ov = [(np.nan, np.nan, 0, np.nan, 'skipped')] * len(df)
for i, row in df.iterrows():
    r = odd_even_check(row['Target'], row['Sector'],
                       row['Period (d)'], row['Duration (hr)'])
    ov[i] = (r['depth_odd'], r['depth_even'], r['n_transits'], r['sigma'], r['reason'])
    tag = 'NaN' if not np.isfinite(r['sigma']) else f"{r['sigma']:.2f}"
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"sigma={tag:<6} n={r['n_transits']:<3} {r['reason']}")

d1, d2, ntr, sig, rsn = zip(*ov)
df['depth_odd_ppm'] = d1
df['depth_even_ppm'] = d2
df['n_transits'] = ntr
df['odd_even_sigma'] = sig
df['odd_even_reason'] = rsn
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_THRESHOLD) if np.isfinite(x) else pd.NA for x in sig],
    dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   通过: {int((df["odd_even_pass"]==True).sum())}  '
      f'不通过: {int((df["odd_even_pass"]==False).sum())}  '
      f'无法判定: {int(df["odd_even_pass"].isna().sum())}')

=== 校验一：奇偶深度 ===
[  1/500] TIC 293694800      sigma=1.04   n=7   ok
[  2/500] TIC 156919710      sigma=0.37   n=3   ok
[  3/500] TIC 232050760      sigma=0.79   n=2   ok
[  4/500] TIC 157077690      sigma=2.46   n=10  ok
[  5/500] TIC 232041402      sigma=NaN    n=1   err_undef(n=1)
[  6/500] TIC 293647739      sigma=0.75   n=37  ok
[  7/500] TIC 737891674      sigma=2.19   n=40  ok
[  8/500] TIC 156836939      sigma=0.00   n=2   ok
[  9/500] TIC 232049607      sigma=20.49  n=3   ok
[ 10/500] TIC 293693365      sigma=0.91   n=5   ok
[ 11/500] TIC 293647910      sigma=1.03   n=7   ok
[ 12/500] TIC 157036437      sigma=1.32   n=3   ok
[ 13/500] TIC 170104965      sigma=1.75   n=9   ok
[ 14/500] TIC 156919054      sigma=NaN    n=2   err_undef(n=2)
[ 15/500] TIC 157014929      sigma=0.17   n=4   ok
[ 16/500] TIC 293775731      sigma=0.86   n=19  ok
[ 17/500] TIC 156886258      sigma=1.51   n=10  ok
[ 18/500] TIC 232041166      sigma=7.19   n=2   ok
[ 19/500] TIC 156920701      sigma=4.23  

## 3. 校验二：行星半径合理性（批量 TIC 查询）

**提速关键**：识别目标名中的 TIC 编号，用 `Catalogs.query_criteria` 一次 SQL 查询拿回所有恒星参数，替代逐条 `query_object`。

In [5]:
from astroquery.mast import Catalogs

print('=== 校验二：行星半径 ===')

# ---- 辅助函数 ----
def _sf(x):
    try: return np.nan if x is None or np.ma.is_masked(x) else float(x)
    except: return np.nan

def _spectral(teff):
    if not np.isfinite(teff): return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T'

# ---- 分离两类目标：可批量查 TIC ID 的 vs 需按名查的 ----
STAR_RAD, STAR_TEFF, STAR_TIC = {}, {}, {}

tic_ids = []
name_targets = []
for _, row in df.iterrows():
    t = str(row['Target']).strip()
    if t.upper().startswith('TIC') and t[3:].strip().isdigit():
        tic_ids.append(int(t[3:].strip()))
    else:
        name_targets.append(t)

print(f'  TIC 编号目标(可批量): {len(tic_ids)}  名称目标(逐个): {len(name_targets)}')

# ---- 批量查 TIC ----
if tic_ids:
    ids_str = ','.join(str(x) for x in tic_ids)
    try:
        batch = Catalogs.query_criteria(
            catalog='TIC', select='ID,rad,Teff,mass,Tmag',
            where=f'ID IN ({ids_str})')
        if batch is not None:
            for r in batch:
                key = f'TIC {int(r["ID"])}'
                STAR_RAD[key] = _sf(r['rad'])
                STAR_TEFF[key] = _sf(r['Teff'])
                STAR_TIC[key] = int(r['ID'])
            print(f'  批量查询命中: {len(STAR_RAD)} 个')
    except Exception as e:
        print(f'  批量查询失败: {e}，将逐条查询...')
        for tid in tic_ids:
            try:
                cat = Catalogs.query_object(f'TIC {tid}', catalog='TIC', radius=0.02)
                if cat is not None and len(cat) > 0:
                    r_ = cat[0]
                    STAR_RAD[f'TIC {tid}'] = _sf(r_['rad'])
                    STAR_TEFF[f'TIC {tid}'] = _sf(r_['Teff'])
                    STAR_TIC[f'TIC {tid}'] = int(r_['ID'])
            except Exception:
                pass

# ---- 按名称逐个查 (少数) ----
for t in name_targets:
    try:
        cat = Catalogs.query_object(t, catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:
                order = np.argsort([_sf(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            STAR_RAD[t] = _sf(r['rad'])
            STAR_TEFF[t] = _sf(r['Teff'])
            STAR_TIC[t] = int(r['ID'])
    except Exception:
        pass

print(f'  总计查询到恒星参数: {len(STAR_RAD)} 个')

=== 校验二：行星半径 ===
  TIC 编号目标(可批量): 500  名称目标(逐个): 0
  批量查询失败: Filter 'select' does not exist. Did you mean '_selected_'?，将逐条查询...
  总计查询到恒星参数: 500 个


In [6]:
# ---- 逐目标判定半径合理性 ----
rj_list, re_list, spec_list, rsn_list = [], [], [], []
tic_list, rstar_list, teff_list = [], [], []
plausible_list = []

for i, row in df.iterrows():
    t = str(row['Target']).strip()
    rad = STAR_RAD.get(t, np.nan)
    te = STAR_TEFF.get(t, np.nan)
    ti = STAR_TIC.get(t, None)
    sp = _spectral(te)
    d = float(row['Depth (ppm)']) * 1e-6

    if not np.isfinite(d) or d <= 0:
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'bad_depth'
    elif not np.isfinite(rad):
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'no_stellar_radius'
    else:
        rj = np.sqrt(d) * rad * R_SUN_TO_RJUP
        rre = rj * R_JUP_TO_REARTH
        mx = MAX_RP_MDWARF if sp == 'M' else MAX_RP_GENERAL
        if rj <= mx:
            ok, reason = True, 'plausible'
        else:
            ok, reason = False, ('too_large_mdwarf' if sp == 'M' else 'too_large')

    rj_list.append(rj); re_list.append(rre); spec_list.append(sp)
    rsn_list.append(reason); tic_list.append(ti)
    rstar_list.append(rad); teff_list.append(te)
    plausible_list.append(ok)

    rt = 'NaN' if not np.isfinite(rj) else f'{rj:.2f}'
    rs = 'NA' if not np.isfinite(rad) else f'{rad:.3f}'
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"R*={rs:<7} 型={sp:<5} Rp={rt:<7}Rjup  {str(ok):<5}")

df['TIC_ID'] = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K'] = teff_list
df['Spectral_Type'] = spec_list
df['Rp_Rjup'] = rj_list
df['Rp_Rearth'] = re_list
df['radius_reason'] = rsn_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   合理: {int((df["planet_radius_plausible"]==True).sum())}  '
      f'不合理: {int((df["planet_radius_plausible"]==False).sum())}  '
      f'无法判定: {int(df["planet_radius_plausible"].isna().sum())}')

[  1/500] TIC 293694800      R*=19.550  型=K     Rp=3.71   Rjup  False
[  2/500] TIC 156919710      R*=8.954   型=K     Rp=3.06   Rjup  False
[  3/500] TIC 232050760      R*=1.727   型=F     Rp=0.67   Rjup  True 
[  4/500] TIC 157077690      R*=18.427  型=K     Rp=6.00   Rjup  False
[  5/500] TIC 232041402      R*=1.302   型=F     Rp=0.50   Rjup  True 
[  6/500] TIC 293647739      R*=1.415   型=A     Rp=0.70   Rjup  True 
[  7/500] TIC 737891674      R*=1.089   型=G     Rp=5.71   Rjup  False
[  8/500] TIC 156836939      R*=9.602   型=K     Rp=3.57   Rjup  False
[  9/500] TIC 232049607      R*=92.624  型=M     Rp=91.69  Rjup  False
[ 10/500] TIC 293693365      R*=3.906   型=K     Rp=0.75   Rjup  True 
[ 11/500] TIC 293647910      R*=2.434   型=F     Rp=0.52   Rjup  True 
[ 12/500] TIC 157036437      R*=9.254   型=K     Rp=3.16   Rjup  False
[ 13/500] TIC 170104965      R*=18.200  型=K     Rp=5.80   Rjup  False
[ 14/500] TIC 156919054      R*=5.122   型=K     Rp=21.44  Rjup  False
[ 15/500] TIC 157014

## 4. 校验三：NASA 档案交叉比对

In [7]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

print('=== 校验三：NASA 档案交叉比对 ===')

_CACHE = {}

def _okt(x):
    try: return x is not None and not pd.isna(x) and int(x) > 0
    except: return False

def crossmatch(target, tic_id=None):
    '''返回 (已知/新候选, 命中来源)。'''
    key = (str(target), int(tic_id) if _okt(tic_id) else None)
    if key in _CACHE:
        return _CACHE[key]
    label, source = '新候选', 'none'
    # (1) 确认行星表：按名称
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label, source = '已知', 'confirmed'
    except Exception:
        source = 'query_error'
    # (2) TOI 表：按 TIC ID
    if label == '新候选' and _okt(tic_id):
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label, source = '已知', 'TOI'
        except Exception:
            if source != 'confirmed':
                source = 'query_error'
    _CACHE[key] = (label, source)
    return (label, source)

labels, sources = [], []
has_tic = 'TIC_ID' in df.columns
for i, row in df.iterrows():
    tic = row['TIC_ID'] if has_tic else None
    lb, so = crossmatch(row['Target'], tic)
    labels.append(lb); sources.append(so)
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} -> {lb:<6} ({so})")

df['已知/新候选'] = labels
df['match_source'] = sources

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   已知: {labels.count("已知")}  新候选: {labels.count("新候选")}')

=== 校验三：NASA 档案交叉比对 ===
[  1/500] TIC 293694800      -> 新候选    (none)
[  2/500] TIC 156919710      -> 新候选    (none)
[  3/500] TIC 232050760      -> 新候选    (none)
[  4/500] TIC 157077690      -> 新候选    (none)
[  5/500] TIC 232041402      -> 新候选    (none)
[  6/500] TIC 293647739      -> 新候选    (none)
[  7/500] TIC 737891674      -> 新候选    (none)
[  8/500] TIC 156836939      -> 新候选    (none)
[  9/500] TIC 232049607      -> 新候选    (none)
[ 10/500] TIC 293693365      -> 新候选    (none)
[ 11/500] TIC 293647910      -> 新候选    (none)
[ 12/500] TIC 157036437      -> 新候选    (none)
[ 13/500] TIC 170104965      -> 新候选    (none)
[ 14/500] TIC 156919054      -> 新候选    (none)
[ 15/500] TIC 157014929      -> 新候选    (none)
[ 16/500] TIC 293775731      -> 新候选    (none)
[ 17/500] TIC 156886258      -> 新候选    (none)
[ 18/500] TIC 232041166      -> 新候选    (none)
[ 19/500] TIC 156920701      -> 新候选    (none)
[ 20/500] TIC 156885811      -> 新候选    (none)
[ 21/500] TIC 156836701      -> 新候选    (none)
[ 22/500] 

## 5. 最终汇总与高置信度清单

In [8]:
# ---- Person1 数据完整性校验 ----
_p = [c for c in P1_COLS if c in df.columns]
assert df[_p].equals(ORIG_DF[_p]), 'Person1 原始数据被意外修改!'
print('✓ Person1 原始数据完整性校验通过\n')

# ---- 高置信度：奇偶通过 + 半径合理 + 非 Person1 原判假阳性 ----
fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
mask = (df['odd_even_pass'] == True) & (df['planet_radius_plausible'] == True) & (~fp)
hc = df[mask.values].copy()

# ---- 按 FINAL_COLS 列序写入最终结果 ----
final = df[[c for c in FINAL_COLS if c in df.columns]].copy()
final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
hc.to_csv('high_confidence_candidates.csv', index=False, encoding='utf-8-sig')

print('=' * 64)
print('三项质控汇总')
print('=' * 64)
print(f'候选总数     : {len(df)}')
print(f'奇偶通过     : {int((df["odd_even_pass"]==True).sum())}')
print(f'半径合理     : {int((df["planet_radius_plausible"]==True).sum())}')
print(f'已知/新候选   : {labels.count("已知")} / {labels.count("新候选")}')
print(f'>> 高置信度  : {len(hc)}  ->  high_confidence_candidates.csv')
print('=' * 64)
final.head(10)

✓ Person1 原始数据完整性校验通过

三项质控汇总
候选总数     : 500
奇偶通过     : 397
半径合理     : 319
已知/新候选   : 1 / 499
>> 高置信度  : 72  ->  high_confidence_candidates.csv


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible,已知/新候选,match_source
0,TIC 293694800,61,3.31785,0.06627,380.4,5.40,6.98,True,400.295454,309.727281,7,1.038166,ok,True,293694800,19.54990,4459.3,K,3.710451,41.590077,too_large,False,新候选,none
1,TIC 156919710,6,6.52283,0.14106,1230.7,2.95,6.76,True,1119.478357,1276.978194,3,0.366310,ok,True,156919710,8.95381,4719.0,K,3.056651,34.261700,too_large,False,新候选,none
2,TIC 232050760,7,18.93983,0.46561,1575.2,1.10,6.18,True,1334.300634,1748.419406,2,0.791259,ok,True,232050760,1.72709,6840.0,F,0.667029,7.476662,plausible,True,新候选,none
3,TIC 157077690,6,2.17228,0.04949,1119.3,6.00,15.63,False,1393.293639,844.213710,10,2.462297,ok,True,157077690,18.42650,4390.0,K,5.998989,67.242064,too_large,False,新候选,none
4,TIC 232041402,6,17.04370,0.52106,1528.9,6.00,9.14,False,0.000000,1438.561098,1,NaN,err_undef(n=1),<NA>,232041402,1.30150,6516.0,F,0.495217,5.550839,plausible,True,新候选,none
5,TIC 293647739,33,0.66591,0.00006,2552.5,0.50,31.25,False,2731.871933,2539.200067,37,0.745266,ok,True,293647739,1.41456,7886.0,A,0.695450,7.795234,plausible,True,新候选,none
6,TIC 737891674,33,0.60534,0.00008,290813.8,1.70,21.31,True,265172.832561,321276.500008,40,2.192368,ok,True,737891674,1.08876,5922.0,G,5.713492,64.041957,too_large,False,新候选,none
7,TIC 156836939,6,10.79439,0.19659,1456.4,6.00,8.68,False,1456.270310,1456.582771,2,0.000826,ok,True,156836939,9.60225,4797.0,K,3.565950,39.970380,too_large,False,新候选,none
8,TIC 232049607,61,11.95313,0.30917,10349.0,1.10,121.93,False,155.877017,2596.543267,3,20.493098,ok,False,232049607,92.62380,3681.0,M,91.692481,1027.771849,too_large_mdwarf,False,新候选,none
9,TIC 293693365,61,3.08347,0.09020,389.7,1.10,5.20,True,490.817354,314.297415,5,0.905647,ok,True,293693365,3.90605,5051.0,K,0.750352,8.410618,plausible,True,新候选,none
